# Landing to Bronze
- Preserva dados de origem, adicionando apenas a coluna `ingestion_datetime` para registrar o momento da ingestão

## 1. Configurações Iniciais

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from datetime import date, datetime, timedelta
import requests

CATALOG = "workspace"
LANDING = "landing"
BRONZE = "bronze"

#Criação das camadas landing/bronze
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{LANDING}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{LANDING}.inputs")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE}")

print(f"Schema {CATALOG}.{LANDING} criado com sucesso ou já existente.")
print(f"Schema {CATALOG}.{BRONZE} criado com sucesso ou já existente.")

Schema workspace.landing criado com sucesso ou já existente.
Schema workspace.bronze criado com sucesso ou já existente.


## 2. Ingestão dos Arquivos

In [0]:
INPUT_PATH = f"/Volumes/{CATALOG}/{LANDING}/inputs/"

#Mapeamento 
FILES_TO_TABLES = {
    "movies_info_TMDB_IMDB.csv" : "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv" : "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv" : "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv" : "tb_credits_and_tags", 
    "movies_reviews.csv" : "tb_movies_reviews"
}

#### 2.1 Função de Ingestão
Padronizar a ingestão dos arquivos CSV da camada Landing para as tabelas correspondentes Bronze. 

In [0]:
def ingest_csv_to_bronze(file_name, table_name):

    file_path = f"{INPUT_PATH}{file_name}"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(file_path)
        .withColumn("ingestion_datetime", F.current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{CATALOG}.{BRONZE}.{table_name}")
    )

    print(
        f"Arquivo '{file_name}' ingerido com sucesso "
        f"na tabela '{CATALOG}.{BRONZE}.{table_name}'."
    )


#### 2.2 Execução da Ingestão

In [0]:
for file_name, table_name in FILES_TO_TABLES.items():
    ingest_csv_to_bronze(file_name, table_name)

Arquivo 'movies_info_TMDB_IMDB.csv' ingerido com sucesso na tabela 'workspace.bronze.tb_movies_info'.
Arquivo 'movies_financials_IMDB_TMDB.csv' ingerido com sucesso na tabela 'workspace.bronze.tb_movies_financials'.
Arquivo 'movies_metrics_IMDB_TMDB.csv' ingerido com sucesso na tabela 'workspace.bronze.tb_movies_metrics'.
Arquivo 'credits_and_tags_IMDB_TMDB.csv' ingerido com sucesso na tabela 'workspace.bronze.tb_credits_and_tags'.
Arquivo 'movies_reviews.csv' ingerido com sucesso na tabela 'workspace.bronze.tb_movies_reviews'.


## 3. Ingestão da API do Banco Central

#### 3.1 Formatação das Datas de Início e Fim

In [0]:
dbutils.widgets.text("data_inicio", "")
dbutils.widgets.text("data_fim", "")

#formato da data
FMT = "%m-%d-%Y"

data_fim = dbutils.widgets.get("data_fim").strip() or date.today().strftime(FMT)
data_inicio = (dbutils.widgets.get("data_inicio").strip() or (date.today() - timedelta(days=7)).strftime(FMT))

#validação
datetime.strptime(data_inicio, FMT); datetime.strptime(data_fim, FMT)

print(f"Data início: {data_inicio}")
print(f"Data fim: {data_fim}")


Data início: 09-12-2026
Data fim: 09-19-2026


#### 3.2 URL da API

In [0]:
API_URL = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

print(API_URL)

https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='09-12-2026'&@dataFinalCotacao='09-19-2026'&$select=dataHoraCotacao,cotacaoCompra&$format=json


#### 3.3 Acesso à API e gravação 

In [0]:
#garante que os tipos vão estar corretos mesmo que a API mude a ordem
schema_cotacao = StructType([
    StructField("dataHoraCotacao", StringType()),
    StructField("cotacaoCompra", DoubleType()),
])

response = requests.get(API_URL, timeout=60)
response.raise_for_status()  # erro HTTP interrompe a execução

dados_cotacao = response.json().get("value", [])

#Validação para quando o retorno for vazio (pegar somente fim de semana)
if not dados_cotacao:
    print(f"Sem cotações entre {data_inicio} e {data_fim}")
else:
    df_cotacao = spark.createDataFrame(
        [(d["dataHoraCotacao"], d["cotacaoCompra"]) for d in dados_cotacao],
        schema_cotacao,
    )
    display(df_cotacao)

    #Ingestão na camada bronze
    df_cotacao = df_cotacao.withColumn("ingestion_datetime", F.current_timestamp())

    (
        df_cotacao.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{CATALOG}.{BRONZE}.tb_cotacao_dolar")
    )
    print("Tabela bronze.tb_cotacao_dolar atualizada com sucesso.")

dataHoraCotacao,cotacaoCompra
2026-09-14 13:10:08.144425,5.169
2026-09-15 13:09:19.199664,5.1484
2026-09-16 13:05:30.35873,5.152
2026-09-17 13:03:21.858212,5.1515
2026-09-18 13:03:34.742036,5.1569


Tabela bronze.tb_cotacao_dolar atualizada com sucesso.


## 4. Validações Finais
Algumas verificações para confirmar se a ingestão da camada Bronze foi concluída corretamente. 

#### 4.1 Validação Tabelas
(Verifica se todas as tabelas esperadas foram criadas no schema `bronze`)

In [0]:
expected_tables = [
    "tb_movies_info",
    "tb_movies_financials",
    "tb_movies_metrics",
    "tb_credits_and_tags",
    "tb_movies_reviews",
    "tb_cotacao_dolar"
]

bronze_tables = [
    table.name
    for table in spark.catalog.listTables(f"{CATALOG}.{BRONZE}")
]

for table in expected_tables:
    if table in bronze_tables:
        print(f"OK - {table}")
    else:
        print(f"ERRO - {table} não encontrada")

OK - tb_movies_info
OK - tb_movies_financials
OK - tb_movies_metrics
OK - tb_credits_and_tags
OK - tb_movies_reviews
OK - tb_cotacao_dolar


#### 4.2 Quantidade de Registros
(Verifica quantidade de registros em cada tabela da camada Bronze)

In [0]:
for table in expected_tables:
    count = spark.table(f"{CATALOG}.{BRONZE}.{table}").count()
    print(f"{table}: {count} registros")

tb_movies_info: 106930 registros
tb_movies_financials: 106165 registros
tb_movies_metrics: 107364 registros
tb_credits_and_tags: 106320 registros
tb_movies_reviews: 32412 registros
tb_cotacao_dolar: 5 registros


#### 4.3 Validação da Estrutura das Tabelas
(Verifica a estrutura, além de confirmar que possuem a coluna "ingestion_datetime") 

In [0]:
errors = []
for t in expected_tables:
    df = spark.table(f"{CATALOG}.{BRONZE}.{t}")
    if "ingestion_datetime" not in df.columns:
        errors.append(f"{t}: sem ingestion_datetime")
    if df.limit(1).count() == 0:
        errors.append(f"{t}: vazia")
        
    #tudo deve ser string, exceto a cotação
    if t != "tb_cotacao_dolar":
        tipados = [f.name for f in df.schema.fields
                   if f.name != "ingestion_datetime" and not isinstance(f.dataType, StringType)]
        if tipados:
            errors.append(f"{t}: colunas não-STRING {tipados}")
if errors:
    raise Exception("Validação Bronze falhou:\n" + "\n".join(errors))
print("Validação Bronze OK")

Validação Bronze OK
